# Day 2-01｜Detection Box Footpoint 與 BEV 投影互動
> Python 籃球運動資料分析課程  
> 本單元先以已訓練 YOLO detector 產生球員框，再調整 bbox 觀察 bottom-center footpoint 對 BEV 投影的影響。  
> 修課背景：具備基礎 Python 語法即可；不預設電腦視覺或運動資料分析經驗。

## 學習目標
- 從模型輸出的 player bbox 取得 bottom-center footpoint。
- 使用 Day 1 Homography 將 footpoint 投影到 BEV。
- 透過互動工具調整 bbox，觀察 BEV 點位變化。


## 課程流程
1. 對球場 frame 執行 YOLO detection。
2. 顯示含有 player box 與 footpoint 的影像。
3. 使用互動 UI 調整 bbox，觀察 footpoint 與 BEV 投影。


In [ ]:
from pathlib import Path
import sys

# Colab 重新啟動 runtime 後，先掛載學生自己的 Google Drive。
try:
    from google.colab import drive  # type: ignore[import-not-found]

    IN_COLAB = True
    drive.mount("/content/drive")
except ModuleNotFoundError:
    IN_COLAB = False

COURSE_ROOT = Path("/content/drive/MyDrive/basketball_hackathon/course")
if not COURSE_ROOT.exists():
    # 本機驗證或 zip 解壓後執行時，從目前資料夾往上找課程根目錄。
    here = Path.cwd().resolve()
    for candidate in [here, *here.parents]:
        if (candidate / "src").exists() and (candidate / "assets").exists():
            COURSE_ROOT = candidate
            break

if not COURSE_ROOT.exists():
    raise FileNotFoundError("找不到課程資料夾，請先執行 init_colab.ipynb。")

if str(COURSE_ROOT) not in sys.path:
    sys.path.insert(0, str(COURSE_ROOT))

from src.course_setup import install_requirements_if_colab, print_environment_summary  # noqa: E402

install_requirements_if_colab(COURSE_ROOT)
print_environment_summary(COURSE_ROOT)


In [ ]:
import numpy as np

from src.cv_utils import (
    bottom_center,
    draw_boxes,
    draw_points,
    read_image_rgb,
    save_image_rgb,
    show_image,
    side_by_side,
)
from src.geometry_utils import compute_homography, default_homography_pairs, project_points, render_bev_court
from src.yolo_utils import detector_model_path, run_detector_on_image

IMAGE_PATH = COURSE_ROOT / "assets" / "samples" / "sample_court_frame.png"
frame = read_image_rgb(IMAGE_PATH)
bev = render_bev_court(COURSE_ROOT / "assets" / "samples" / "sample_bev_court.json")

pairs = default_homography_pairs()
H = compute_homography(
    [pair["camera_xy"] for pair in pairs],
    [pair["bev_xy"] for pair in pairs],
)

detections, _ = run_detector_on_image(
    detector_model_path(COURSE_ROOT),
    frame,
    conf=0.25,
    imgsz=960,
)
players = [d for d in detections if d.class_name == "player"]
if not players:
    raise RuntimeError("此 frame 沒有偵測到 player，請調整 conf 或更換影像。")

player = players[0]
bbox = player.bbox_xyxy
foot = bottom_center(bbox)
bev_foot = project_points([foot], H)[0]

frame_vis = draw_boxes(frame, [bbox], [f"player {player.confidence:.2f}"])
frame_vis = draw_points(frame_vis, [foot], ["footpoint"])
bev_vis = draw_points(bev, [bev_foot], ["projected"])
combo = side_by_side(frame_vis, bev_vis)
show_image(combo, "detected bbox and projected footpoint")

out_path = COURSE_ROOT / "assets" / "results" / "d2_01_detected_bbox_to_bev.png"
save_image_rgb(out_path, combo)
print("bbox:", [round(v, 2) for v in bbox])
print("footpoint:", [round(v, 2) for v in foot])
print("BEV:", np.round(bev_foot, 2).tolist())
print("saved:", out_path)


In [ ]:
from src.notebook_widgets import show_bbox_to_bev_tool

show_bbox_to_bev_tool(
    IMAGE_PATH,
    initial_bbox=bbox,
    homography_matrix=H.tolist(),
    bev_width=bev.shape[1],
    bev_height=bev.shape[0],
    canvas_width=1000,
    title="調整 player bbox 並觀察 BEV footpoint",
)


課堂操作：調整 `x1`、`y1`、`x2`、`y2`。bbox 底邊中心點越接近球員真正的地面接觸位置，BEV 投影越穩定。